In [1]:
import os
import time
import pandas as pd
from nba_api.stats.endpoints import leaguegamefinder, boxscoretraditionalv2
from requests.exceptions import ReadTimeout, ConnectionError

# 定义计算 Possessions 的函数
def calculate_possessions(fga, fta, orb, tov, opp_fga, opp_fta, opp_orb, opp_tov):
    possessions = 0.5 * ((fga + 0.4 * fta - orb + tov) + (opp_fga + 0.4 * opp_fta - opp_orb + opp_tov))
    return possessions

# 获取 2023-24 赛季的常规赛比赛信息
def get_all_games(season='2023-24'):
    print("Fetching all games...")
    gamefinder = leaguegamefinder.LeagueGameFinder(season_nullable=season, season_type_nullable='Regular Season')
    games = gamefinder.get_data_frames()[0]

    # 打印列名，确保正确
    print("Columns in the games data:", games.columns)

    # 过滤仅包含 2023-24 赛季的常规赛
    games = games[games['SEASON_ID'].astype(str).str.endswith('2023')]

    # 删除重复的比赛记录（基于 GAME_ID）
    games = games.drop_duplicates(subset=['GAME_ID'])

    # 过滤掉异常的MATCHUP（只保留NBA常规赛球队）
    valid_teams = ['ATL', 'BOS', 'BKN', 'CHA', 'CHI', 'CLE', 'DAL', 'DEN', 'DET', 
                   'GSW', 'HOU', 'IND', 'LAC', 'LAL', 'MEM', 'MIA', 'MIL', 'MIN', 
                   'NOP', 'NYK', 'OKC', 'ORL', 'PHI', 'PHX', 'POR', 'SAC', 'SAS', 
                   'TOR', 'UTA', 'WAS']
    games = games[games['MATCHUP'].str[:3].isin(valid_teams)]

    print(f"Filtered regular season games count: {len(games)}")
    return games


# 获取比赛统计数据并计算 possessions，同时存储数据
def get_game_stats_and_possessions(games, output_file='game_data.csv'):
    # 检查是否已有保存数据，避免重复抓取
    if os.path.exists(output_file):
        saved_data = pd.read_csv(output_file)
        saved_game_ids = set(saved_data['GAME_ID'])
        print(f"{len(saved_game_ids)} games already saved. Skipping these games...")
    else:
        saved_data = pd.DataFrame()
        saved_game_ids = set()

    all_game_data = []
    for _, row in games.iterrows():
        game_id = row['GAME_ID']

        if game_id in saved_game_ids:
            continue  # 跳过已保存的比赛

        try:
            for attempt in range(3):  # 最多尝试3次
                try:
                    time.sleep(1)  # 添加延迟，防止触发限流
                    boxscore = boxscoretraditionalv2.BoxScoreTraditionalV2(game_id=game_id, timeout=60)
                    break  # 请求成功，跳出重试循环
                except (ReadTimeout, ConnectionError) as e:
                    print(f"Error fetching game {game_id}: {e}. Retrying... (Attempt {attempt + 1})")
                    time.sleep(5)  # 失败后等待 5 秒再尝试
            else:
                print(f"Failed to fetch data for game {game_id} after 3 attempts.")
                continue

            # 获取 team stats 数据（两支球队的统计数据）
            game_data = boxscore.get_data_frames()[1]
            team_stats = game_data[['TEAM_ID', 'FGA', 'FTA', 'OREB', 'TO', 'PTS']]

            # 分割成两队数据
            team1 = team_stats.iloc[0]
            team2 = team_stats.iloc[1]

            # 计算 Possessions
            possessions = calculate_possessions(
                team1['FGA'], team1['FTA'], team1['OREB'], team1['TO'],
                team2['FGA'], team2['FTA'], team2['OREB'], team2['TO']
            )

            # 将比赛信息转换为字典，并添加 Possessions
            row_data = row.to_dict()
            row_data['POSSESSIONS'] = possessions
            all_game_data.append(row_data)

            # 立即将数据保存到 CSV，防止中途出错丢失数据
            pd.DataFrame([row_data]).to_csv(output_file, mode='a', header=not os.path.exists(output_file), index=False)

        except Exception as e:
            print(f"Error processing game {game_id}: {e}")

    return pd.DataFrame(all_game_data)

# 批量处理比赛，防止一次性请求过多导致失败
def process_games_in_batches(games, batch_size=100, output_file='game_data.csv'):
    total_games = len(games)
    all_game_data = []
    for start_idx in range(0, total_games, batch_size):
        end_idx = min(start_idx + batch_size, total_games)
        batch_games = games.iloc[start_idx:end_idx]
        print(f"Processing games {start_idx + 1} to {end_idx}...")
        batch_data = get_game_stats_and_possessions(batch_games, output_file=output_file)
        all_game_data.append(batch_data)
        time.sleep(10)  # 每批次请求完成后暂停 10 秒，防止限流
    return pd.concat(all_game_data, ignore_index=True)

# 主程序执行
if __name__ == "__main__":
    output_file = 'game_data.csv'  # 保存的本地文件
    games = get_all_games()  # 获取所有常规赛比赛
    possession_data = process_games_in_batches(games, output_file=output_file)
    print(f"Data saved to {output_file}")


Fetching all games...


ProxyError: HTTPSConnectionPool(host='stats.nba.com', port=443): Max retries exceeded with url: /stats/leaguegamefinder?Conference=&DateFrom=&DateTo=&Division=&DraftNumber=&DraftRound=&DraftTeamID=&DraftYear=&EqAST=&EqBLK=&EqDD=&EqDREB=&EqFG3A=&EqFG3M=&EqFG3_PCT=&EqFGA=&EqFGM=&EqFG_PCT=&EqFTA=&EqFTM=&EqFT_PCT=&EqMINUTES=&EqOREB=&EqPF=&EqPTS=&EqREB=&EqSTL=&EqTD=&EqTOV=&GameID=&GtAST=&GtBLK=&GtDD=&GtDREB=&GtFG3A=&GtFG3M=&GtFG3_PCT=&GtFGA=&GtFGM=&GtFG_PCT=&GtFTA=&GtFTM=&GtFT_PCT=&GtMINUTES=&GtOREB=&GtPF=&GtPTS=&GtREB=&GtSTL=&GtTD=&GtTOV=&LeagueID=&Location=&LtAST=&LtBLK=&LtDD=&LtDREB=&LtFG3A=&LtFG3M=&LtFG3_PCT=&LtFGA=&LtFGM=&LtFG_PCT=&LtFTA=&LtFTM=&LtFT_PCT=&LtMINUTES=&LtOREB=&LtPF=&LtPTS=&LtREB=&LtSTL=&LtTD=&LtTOV=&Outcome=&PORound=&PlayerID=&PlayerOrTeam=T&RookieYear=&Season=2023-24&SeasonSegment=&SeasonType=Regular+Season&StarterBench=&TeamID=&VsConference=&VsDivision=&VsTeamID=&YearsExperience= (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 407 

In [ ]:
#0022300605

In [5]:
# Smart Proxy 配置
PROXY_HOST = "us.smartproxy.com"  # 代理服务器地址
PROXY_PORT = "10000"  # 代理端口（请根据你的Smart Proxy提供的端口修改）
PROXY_USER = "U0000232037"  # 你的用户名
PROXY_PASS = "PW_19d9245e5bf29aa31bee16e0c76987f99"  # 你的密码

# 构建代理URL
proxy_url = f"http://{PROXY_USER}:{PROXY_PASS}@{PROXY_HOST}:{PROXY_PORT}"

# 配置代理
proxies = {
    "http": proxy_url,
    "https": proxy_url
}

import requests
session = requests.Session()
session.proxies.update(proxies)

# 将 session 注入到 nba_api
from nba_api.stats.library.http import NBAStatsHTTP
NBAStatsHTTP._session = session

# 测试请求
response = session.get("https://stats.nba.com/stats/leaguegamefinder?Season=2023-24&SeasonType=Regular+Season")
print(response.status_code, response.text[:500])


ProxyError: HTTPSConnectionPool(host='stats.nba.com', port=443): Max retries exceeded with url: /stats/leaguegamefinder?Season=2023-24&SeasonType=Regular+Season (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 407 Proxy Authentication Required')))